# Redundant, or just diluted?

Adding all 60 calibrated features to the 50 base gave **-0.0013 (t -0.25)** — no gain — yet the
model spent **46.5% of its importance** on them (`cal_ar5_var_scan` ranked #4 overall). Two
explanations, opposite implications:

- **Redundancy** — the battery repackages signal the 50 already carry. Then no subset helps and
  the family is closed.
- **Dilution** — 60 extra columns against 263k training rows spreads the splits too thin. Then a
  focused subset of the strongest few SHOULD help.

Test: base50 + the top-k calibrated features only (k = 4, 8, 16), same folds, same ramp target,
paired vs `ramp_base50`. Features are already cached so this is cheap.

Note the top-k list is taken from the importance ranking fitted on the SAME data — mild
selection bias, so a marginal win here would need a clean 10-fold re-test before being believed.
A clear null, on the other hand, settles it.

In [1]:
import os, sys, json, time
import numpy as np, pandas as pd
from catboost import CatBoostRegressor

TOOLS = os.path.abspath('../tools'); sys.path.insert(0, TOOLS)
import cv_tools as CV
REPO = os.path.abspath('../../..')

cache = np.load(os.path.join(TOOLS, 'cv_folds_by_series', 'cv_cache_full.npz'))
Xbase, y, sid, tim = cache['X'], cache['y'], cache['sid'], cache['time']
sampled, base_cols = cache['is_sampled_step'], list(cache['cols'])
cc = np.load('calib_features.npz')
Xcal, cal_cols = np.nan_to_num(cc['X'], nan=0.0, posinf=0.0, neginf=0.0), list(cc['cols'])
assert np.array_equal(cc['sid'], sid) and np.array_equal(cc['time'], tim)
index = pd.MultiIndex.from_arrays([sid, tim], names=['id', 'time'])
step = CV.steps_from_index(index)

idx = pd.read_parquet(os.path.join(REPO, 'y_train_index.parquet'))
tau_row = idx['tau_index'].reindex(np.unique(sid)).reindex(sid).to_numpy()
since = step - tau_row; is_break = tau_row >= 0
W = 40
ramp = np.clip(since / (2.0 * W), 0.0, 1.0)
ramp[~is_break] = 0.0; ramp[is_break & (since < 0)] = 0.0
ramp_by_row = pd.Series(ramp, index=index)

# top calibrated features by the importance ranking from calibrated_battery.ipynb
TOPK = ['cal_ar5_var_scan', 'cal_ring12_scanpeak', 'cal_sign_runs_scanpeak', 'cal_ar5_var_exp',
        'cal_rank_disp_scanpeak', 'cal_ar5_var_scanpeak', 'cal_autocorr1_scanpeak',
        'cal_mean_scanpeak', 'cal_ring0_scanpeak', 'cal_jump_scanpeak', 'cal_ar5_var_seg',
        'cal_absdev_scanpeak', 'cal_glr_peak', 'cal_glr_strength', 'cal_var_scanpeak',
        'cal_tail_scanpeak']
TOPK = [c for c in TOPK if c in cal_cols]
colpos = {c: i for i, c in enumerate(cal_cols)}
print(f'{len(TOPK)} candidate top features available')

16 candidate top features available


In [2]:
REG = dict(iterations=1500, learning_rate=0.02, depth=6, l2_leaf_reg=3.0,
           bootstrap_type='Bernoulli', subsample=0.8, loss_function='RMSE',
           random_seed=0, verbose=0, thread_count=-1)
def fp(train, val):
    m = CatBoostRegressor(**REG)
    m.fit(train.X, ramp_by_row.loc[train.index].to_numpy(), sample_weight=train.w)
    return m.predict(val.X)

folds = CV.series_folds(sid, n_splits=5, seed=0)
OUT = 'subset_results.json'
done = json.load(open(OUT)) if os.path.exists(OUT) else {}
res = {}

configs = [('ramp_base50', Xbase)]
for k in (4, 8, 16):
    cols = TOPK[:k]
    M = np.hstack([Xbase, Xcal[:, [colpos[c] for c in cols]]]).astype(np.float32)
    configs.append((f'base50_plus_top{k}', M))

for name, M in configs:
    t = time.time(); print(f'\n=== {name} ({M.shape[1]} feats) ===', flush=True)
    r = CV.run_cv(M, y, sid, index, fp, folds=folds, train_row_mask=sampled,
                  row_step=step, verbose=False)
    res[name] = r
    done[name] = r.fold_scores.tolist(); json.dump(done, open(OUT, 'w'), indent=2)
    print(f'{r.summary(name)} | {time.time()-t:.0f}s', flush=True)


=== ramp_base50 (50 feats) ===


ramp_base50        mean 0.6089 +/- 0.0157 (sem 0.0070) | OOF 0.6079 | folds: 0.5940  0.6044  0.5958  0.6206  0.6297 | 176s



=== base50_plus_top4 (54 feats) ===


base50_plus_top4   mean 0.6119 +/- 0.0109 (sem 0.0049) | OOF 0.6112 | folds: 0.6063  0.6083  0.5988  0.6204  0.6256 | 185s



=== base50_plus_top8 (58 feats) ===


base50_plus_top8   mean 0.6107 +/- 0.0101 (sem 0.0045) | OOF 0.6099 | folds: 0.6040  0.6100  0.5982  0.6232  0.6179 | 187s



=== base50_plus_top16 (66 feats) ===


base50_plus_top16  mean 0.6069 +/- 0.0100 (sem 0.0045) | OOF 0.6061 | folds: 0.6023  0.6043  0.5987  0.6242  0.6048 | 209s


In [3]:
print(CV.summarize(res))
print('\n--- each subset vs base50 (ramp target) ---')
for n in res:
    if n != 'ramp_base50':
        print(CV.paired_compare(res[n], res['ramp_base50'], n, 'ramp_base50'))

base50_plus_top4   mean 0.6119 +/- 0.0109 (sem 0.0049) | OOF 0.6112 | folds: 0.6063  0.6083  0.5988  0.6204  0.6256
base50_plus_top8   mean 0.6107 +/- 0.0101 (sem 0.0045) | OOF 0.6099 | folds: 0.6040  0.6100  0.5982  0.6232  0.6179
ramp_base50        mean 0.6089 +/- 0.0157 (sem 0.0070) | OOF 0.6079 | folds: 0.5940  0.6044  0.5958  0.6206  0.6297
base50_plus_top16  mean 0.6069 +/- 0.0100 (sem 0.0045) | OOF 0.6061 | folds: 0.6023  0.6043  0.5987  0.6242  0.6048

--- each subset vs base50 (ramp target) ---
base50_plus_top4 - ramp_base50: +0.0030 +/- 0.0060 (sem 0.0027, t +1.10, wins 3/5) -> within noise
  per-fold deltas: +0.0123  +0.0039  +0.0030  -0.0002  -0.0040
base50_plus_top8 - ramp_base50: +0.0018 +/- 0.0081 (sem 0.0036, t +0.48, wins 4/5) -> within noise
  per-fold deltas: +0.0099  +0.0056  +0.0024  +0.0026  -0.0118
base50_plus_top16 - ramp_base50: -0.0020 +/- 0.0131 (sem 0.0059, t -0.35, wins 3/5) -> within noise
  per-fold deltas: +0.0083  -0.0001  +0.0029  +0.0036  -0.0248
